# Turnkey perplexity filter
Run the CPU-safe prompt-perplexity example, then inspect its typed-provider accounting.

In [ ]:
import json
import subprocess
import sys
import tempfile
from pathlib import Path
import yaml

workspace = Path(tempfile.mkdtemp(prefix='turnkey-perplexity-'))
config = yaml.safe_load(Path('configs/runs/smoke_perplexity.yaml').read_text())
config['run']['out_dir'] = str(workspace / 'outputs')
config_path = workspace / 'run.yaml'
config_path.write_text(yaml.safe_dump(config, sort_keys=False))
completed = subprocess.run(
    [sys.executable, '-m', 'turnkey.cli', 'run', '--config', str(config_path)],
    check=True, capture_output=True, text=True,
)
run_dir = Path(completed.stdout.strip().splitlines()[-1])
run_dir

In [ ]:
events = [json.loads(line) for line in (run_dir / 'events.jsonl').read_text().splitlines()]
provider_events = [event for event in events if 'provider' in json.dumps(event).lower()]
audit = subprocess.run(
    [sys.executable, '-m', 'turnkey.cli', 'audit', str(run_dir)],
    check=True, capture_output=True, text=True,
)
assert audit.stdout.strip() == 'OK'
{'provider_events': len(provider_events), 'audit': 'passed'}